In [ ]:
# Run once in a fresh environment if required:
# %pip install torch torchvision tensorflow scikit-learn matplotlib seaborn grad-cam
import random, time, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using", DEVICE)
except ImportError:
    print("PyTorch is not installed yet. Run the installation line above.")

In [ ]:
# Question 1. NumPy 2D convolution
# Brief: Apply zero padding, stride-controlled sliding windows, and sum of element-wise products.

In [ ]:
def conv2d(image, kernel, stride=1, padding=0):
    if stride < 1 or padding < 0: raise ValueError("stride must be positive and padding non-negative")
    padded = np.pad(np.asarray(image), ((padding,padding),(padding,padding)), mode="constant")
    kh,kw=kernel.shape; oh=(padded.shape[0]-kh)//stride+1; ow=(padded.shape[1]-kw)//stride+1
    output=np.zeros((oh,ow),dtype=float)
    for row in range(oh):
        for col in range(ow):
            patch=padded[row*stride:row*stride+kh,col*stride:col*stride+kw]
            output[row,col]=np.sum(patch*kernel)
    return padded,output
X=np.array([[1,2,3,0,1],[0,1,2,3,1],[1,2,1,0,0],[2,1,0,1,2],[1,0,2,3,1]])
K=np.array([[1,0,-1],[1,0,-1],[1,0,-1]])
padded,feature=conv2d(X,K,stride=2,padding=1)
print("Padded image:\n",padded); print("\nOutput feature map:\n",feature)

In [ ]:
# Question 2. RNN, LSTM, and GRU for IMDB sentiment
# Brief: Train matched one-layer recurrent classifiers for five epochs, plot validation curves together, measure timing, and evaluate BCE, accuracy, and F1.

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import f1_score

vocab_size, max_len = 10_000, 200
(x_train,y_train),(x_test,y_test)=imdb.load_data(num_words=vocab_size)
x_train=pad_sequences(x_train,maxlen=max_len); x_test=pad_sequences(x_test,maxlen=max_len)
def recurrent_model(kind):
    recurrent={"RNN":tf.keras.layers.SimpleRNN,"LSTM":tf.keras.layers.LSTM,"GRU":tf.keras.layers.GRU}[kind]
    model=tf.keras.Sequential([tf.keras.layers.Embedding(vocab_size,128),recurrent(64),tf.keras.layers.Dense(1,activation="sigmoid")])
    model.compile(optimizer=tf.keras.optimizers.Adam(.001),loss="binary_crossentropy",metrics=["accuracy"]); return model
histories={}; report={}
for kind in ["RNN","LSTM","GRU"]:
    model=recurrent_model(kind); start=time.perf_counter(); history=model.fit(x_train,y_train,validation_split=.2,epochs=5,batch_size=128,verbose=2); seconds=(time.perf_counter()-start)/5
    prob=model.predict(x_test,verbose=0).ravel(); report[kind]={"test_bce":model.evaluate(x_test,y_test,verbose=0)[0],"accuracy":((prob>=.5)==y_test).mean(),"f1":f1_score(y_test,prob>=.5),"seconds_per_epoch":seconds}; histories[kind]=history.history
report

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
for name,h in histories.items(): ax[0].plot(h["val_accuracy"],label=name); ax[1].plot(h["val_loss"],label=name)
ax[0].set(title="Validation accuracy",xlabel="Epoch"); ax[1].set(title="Validation loss",xlabel="Epoch")
for a in ax: a.legend(); a.grid(alpha=.25)
plt.show()

**Interpretation.** Vanilla RNN repeatedly multiplies the recurrent Jacobian, so distant evidence can vanish or explode. LSTM's forget/input/output gates preserve and selectively expose a cell state; GRU's update gate interpolates old and new state while its reset gate controls past information. Both should improve long-review learning. Compare the measured `seconds_per_epoch`: GRU often trains faster because it has fewer gates/parameters than LSTM.

In [ ]:
# Question 3. Multi-head self-attention
# Brief: Implement scaled dot-product attention, split Q/K/V into heads, and print all requested intermediates.

In [ ]:
import torch, torch.nn as nn
class ScaledDotAttention(nn.Module):
    def forward(self,Q,K,V):
        scores=Q@K.transpose(-2,-1)/math.sqrt(Q.size(-1)); weights=torch.softmax(scores,dim=-1); return weights@V,scores,weights
class MultiHeadAttention(nn.Module):
    def __init__(self,d_model,num_heads):
        super().__init__(); assert d_model%num_heads==0; self.h=num_heads; self.dk=d_model//num_heads; self.q=nn.Linear(d_model,d_model); self.k=nn.Linear(d_model,d_model); self.v=nn.Linear(d_model,d_model); self.out=nn.Linear(d_model,d_model); self.attn=ScaledDotAttention()
    def split(self,x): return x.view(x.size(0),x.size(1),self.h,self.dk).transpose(1,2)
    def forward(self,x):
        Q,K,V=self.q(x),self.k(x),self.v(x); Qh,Kh,Vh=map(self.split,(Q,K,V)); z,scores,weights=self.attn(Qh,Kh,Vh); z=z.transpose(1,2).contiguous().view(x.size(0),x.size(1),-1); return self.out(z),{"Q":Q,"K":K,"V":V,"Q_heads":Qh,"K_heads":Kh,"V_heads":Vh,"scores":scores,"weights":weights}
torch.manual_seed(42); X=torch.randn(1,4,8); mha=MultiHeadAttention(8,2); output,info=mha(X)
for name,value in info.items(): print(name,value.detach()); print()
print("Final output:\n",output.detach())

In [ ]:
# Question 4. Causal masked multi-head attention
# Brief: Add a lower-triangular causal mask before softmax and assert that no head attends to a future token.

In [ ]:
class MaskedScaledDotAttention(nn.Module):
    def forward(self,Q,K,V):
        scores=Q@K.transpose(-2,-1)/math.sqrt(Q.size(-1)); mask=torch.tril(torch.ones(scores.size(-2),scores.size(-1),device=scores.device,dtype=torch.bool)); scores=scores.masked_fill(~mask,float("-inf")); weights=torch.softmax(scores,-1); return weights@V,scores,weights,mask
class MaskedMultiHeadAttention(MultiHeadAttention):
    def __init__(self,d_model,num_heads): super().__init__(d_model,num_heads); self.attn=MaskedScaledDotAttention()
    def forward(self,x):
        Q,K,V=self.q(x),self.k(x),self.v(x); Qh,Kh,Vh=map(self.split,(Q,K,V)); z,s,w,mask=self.attn(Qh,Kh,Vh); z=z.transpose(1,2).contiguous().view(x.size(0),x.size(1),-1); return self.out(z),{"Q":Q,"K":K,"V":V,"Q_heads":Qh,"K_heads":Kh,"V_heads":Vh,"mask":mask,"scores":s,"weights":w}
torch.manual_seed(42); X=torch.randn(1,4,8); causal=MaskedMultiHeadAttention(8,2); out,info=causal(X)
for name,value in info.items(): print(name,"\n",value.detach() if torch.is_tensor(value) else value)
future=torch.triu(torch.ones(4,4,dtype=torch.bool),diagonal=1); assert torch.all(info["weights"][:,:,future] == 0), "A future attention weight was non-zero"
print("Final output:\n",out.detach(),"\nAssertion passed: future attention is zero.")

In [ ]:
# Question 5. Why scale attention scores?
# Brief: Visualize unscaled and scaled weights at three key dimensions.

In [ ]:
def attention_scaling_figure(dk,seq_len=4):
    torch.manual_seed(42); Q,K=torch.randn(seq_len,dk),torch.randn(seq_len,dk); raw=Q@K.T; matrices=[torch.softmax(raw,-1),torch.softmax(raw/math.sqrt(dk),-1)]
    fig,axes=plt.subplots(2,2,figsize=(9,7))
    for j,(label,m) in enumerate(zip(["Without scaling","With scaling"],matrices)):
        sns.heatmap(m.numpy(),annot=True,fmt=".2f",vmin=0,vmax=1,ax=axes[0,j],cmap="mako"); axes[0,j].set_title(f"{label}, dk={dk}")
        axes[1,j].hist(m.flatten().numpy(),bins=12); axes[1,j].set(xlabel="Attention weight",ylabel="Count")
    plt.tight_layout(); plt.show()
for dk in [4,16,64]: attention_scaling_figure(dk)

As (d_k) grows, unscaled dot-product variance grows, softmax saturates, and a few near-one weights dominate. Dividing by (sqrt{d_k}) keeps score variance roughly comparable, avoiding saturated softmax gradients and stabilizing optimization.

In [ ]:
# Question 6. Grad-CAM for MNIST
# Brief: Train the specified CNN, visualize correct and incorrect predictions with Grad-CAM overlays, then inspect an occluded input.

In [ ]:
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__(); self.conv1=nn.Conv2d(1,16,3); self.conv2=nn.Conv2d(16,32,3); self.pool=nn.MaxPool2d(2); self.fc1=nn.Linear(32*5*5,128); self.fc2=nn.Linear(128,10)
    def forward(self,x): x=torch.relu(self.conv1(x)); x=self.pool(x); x=torch.relu(self.conv2(x)); x=self.pool(x); return self.fc2(torch.relu(self.fc1(x.flatten(1))))
def train_mnist(model,loader,epochs=5):
    opt=torch.optim.Adam(model.parameters(),lr=.001); ce=nn.CrossEntropyLoss(); model.train()
    for e in range(epochs):
        for x,y in loader: opt.zero_grad(); loss=ce(model(x.to(DEVICE)),y.to(DEVICE)); loss.backward(); opt.step()
        print("epoch",e+1,"done")
def gradcam(model,image,target):
    activations=[]; gradients=[]
    hook1=model.conv2.register_forward_hook(lambda m,i,o: activations.append(o))
    hook2=model.conv2.register_full_backward_hook(lambda m,gi,go: gradients.append(go[0]))
    model.zero_grad(); score=model(image.to(DEVICE))[0,target]; score.backward(); hook1.remove(); hook2.remove()
    weights=gradients[0].mean((2,3),keepdim=True); cam=torch.relu((weights*activations[0]).sum(1,keepdim=True)); cam=torch.nn.functional.interpolate(cam,size=image.shape[-2:],mode="bilinear",align_corners=False)[0,0]; return (cam/cam.max()).detach().cpu()
def show_cam(image,cam,title):
    fig,ax=plt.subplots(); ax.imshow(image[0,0],cmap="gray"); ax.imshow(cam,cmap="jet",alpha=.45); ax.set_title(title); ax.axis("off")
# train_data=datasets.MNIST("data",train=True,download=True,transform=transforms.ToTensor()); test_data=datasets.MNIST("data",train=False,download=True,transform=transforms.ToTensor())
# mnist=MNISTCNN().to(DEVICE); train_mnist(mnist,DataLoader(train_data,128,shuffle=True),5)
# image,label=test_data[0]; prediction=mnist(image.unsqueeze(0).to(DEVICE)).argmax(1).item(); show_cam(image,gradcam(mnist,image.unsqueeze(0),prediction),f"true={label}, pred={prediction}")
# occluded=image.clone(); occluded[:,:,10:18]=0; show_cam(occluded,gradcam(mnist,occluded.unsqueeze(0),prediction),"Occluded digit")

**Analysis.** Correct examples should highlight discriminative strokes (for example, the vertical bar of 1 versus the loops of 8). Misclassifications may highlight ambiguous/shared strokes or noise, revealing why evidence favoured the wrong logit. Occlusion should shift or reduce attention over removed evidence; a sharp confidence/map change signals limited robustness.

In [ ]:
# Question 7. BPTT derivation
# Brief: Derive output, recurrent, and hidden-state gradients for a three-step tanh RNN.

Define the output error at time t as `e_t = y_hat_t - y_t`. Since `y_hat_t = W_hy @ h_t + b_y`, the gradient for the output matrix is:

`dL/dW_hy = sum from t = 1 to 3 of (e_t @ h_t.T)`

Let `a_t = W_hh @ h_(t-1) + W_xh @ x_t + b_h` and `h_t = tanh(a_t)`. Working backward, set the boundary value `delta_4 = 0`. The pre-activation gradient at each step is:

`delta_t = (W_hy.T @ e_t + W_hh.T @ delta_(t+1)) * (1 - h_t^2)`

Here `*` is element-wise multiplication. The recurrent-weight gradient is:

`dL/dW_hh = sum from t = 1 to 3 of (delta_t @ h_(t-1).T)`

During BPTT, gradients repeatedly multiply by recurrent weight/Jacobian terms. Factors with magnitude below one shrink the signal (vanishing gradients), while factors above one amplify it (exploding gradients).

In [ ]:
# Question 8. VGG-like Fashion-MNIST classifier
# Brief: Build a non-pretrained three-block VGG-style CNN with BatchNorm/Dropout, train for at least 15 epochs, and chart accuracy/loss/time.

In [ ]:
class FashionVGG(nn.Module):
    def __init__(self):
        super().__init__()
        def block(a,b): return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(),nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(),nn.MaxPool2d(2))
        self.features=nn.Sequential(block(1,32),block(32,64),block(64,128)); self.head=nn.Sequential(nn.Flatten(),nn.Linear(128*3*3,256),nn.ReLU(),nn.Dropout(.4),nn.Linear(256,10))
    def forward(self,x): return self.head(self.features(x))
def train_fashion(model,train_loader,test_loader,epochs=15):
    opt=torch.optim.Adam(model.parameters(),lr=.001); ce=nn.CrossEntropyLoss(); hist={"train_loss":[],"test_loss":[],"train_acc":[],"test_acc":[],"seconds":[]}
    for e in range(epochs):
        start=time.perf_counter(); model.train(); total=correct=0; loss_sum=0
        for x,y in train_loader:
            opt.zero_grad(); z=model(x.to(DEVICE)); loss=ce(z,y.to(DEVICE)); loss.backward(); opt.step(); loss_sum+=loss.item()*y.size(0); correct+=(z.argmax(1)==y.to(DEVICE)).sum().item(); total+=y.size(0)
        model.eval(); tl=tc=tn=0
        with torch.no_grad():
            for x,y in test_loader: z=model(x.to(DEVICE)); tl+=ce(z,y.to(DEVICE)).item()*y.size(0); tc+=(z.argmax(1)==y.to(DEVICE)).sum().item(); tn+=y.size(0)
        hist["train_loss"].append(loss_sum/total); hist["train_acc"].append(correct/total); hist["test_loss"].append(tl/tn); hist["test_acc"].append(tc/tn); hist["seconds"].append(time.perf_counter()-start); print(e+1,hist["train_acc"][-1],hist["test_acc"][-1],hist["seconds"][-1])
    return hist
# fashion_tf=transforms.ToTensor(); ftrain=datasets.FashionMNIST("data",train=True,download=True,transform=fashion_tf); ftest=datasets.FashionMNIST("data",train=False,download=True,transform=fashion_tf)
# fashion=FashionVGG().to(DEVICE); print("Trainable parameters:",sum(p.numel() for p in fashion.parameters() if p.requires_grad)); fashion_hist=train_fashion(fashion,DataLoader(ftrain,128,shuffle=True),DataLoader(ftest,256),15)
# for key in ["train_acc","test_acc","train_loss","test_loss"]: plt.plot(fashion_hist[key],label=key)
# plt.legend(); plt.xlabel("Epoch"); plt.show(); print("Final train/test accuracy",fashion_hist["train_acc"][-1],fashion_hist["test_acc"][-1],"mean seconds/epoch",np.mean(fashion_hist["seconds"]))